# Lou et al. (arXiv 2610.20067) Table 1 재현 — NQ-Swap · Llama-3.1-8B-Instruct · layer 10

목표 수치(논문 Table 1, NQ Llama): Unsteered **33.7 / 49.4**, Ours **Acc_par 56.8 / Acc_ctx 55.5**.

- `SOTA/lou/upstream`(원 저자 코드)은 수정하지 않고 `SOTA/lou/scripts/`에서 호출만 합니다. 경로·인자 확인 내용은 [SOTA/lou/README.md](README.md)에 있습니다.
- 준비물: Colab Secrets의 `HF_TOKEN`(Llama 3.1 라이선스 승인 계정, 노트북 접근 허용), Drive 여유 공간 약 20 GB(모델 16 GB + dump), GPU L4 또는 A100.
- 단계별 셀은 재실행 시 완료된 산출물을 건너뜁니다. 런타임이 끊기면 설정 셀부터 다시 실행하고 이어서 진행하세요.

## 1. GPU 확인

In [1]:
!nvidia-smi

Thu Nov 26 06:59:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Drive 연결

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. 설정 · 환경 변수

모든 산출물은 Drive의 `sota_lou/` 아래에 저장됩니다. 저장소는 속도를 위해 로컬 `/content`에 clone합니다.

`BATCH_SIZE`: 논문 값은 64(A100 80GB)입니다. 메모리에 따라 자동으로 줄이며, batch 크기는 left padding과 bf16 수치 때문에 greedy 출력을 미세하게 바꿀 수 있습니다(README 참고).

In [3]:
import os, torch
from google.colab import userdata

REPO_URL = "https://github.com/Ontology0/Graduation-Project.git"
BRANCH = "sota/lou"
REPO_DIR = "/content/Graduation-Project"
LOU_DIR = f"{REPO_DIR}/SOTA/lou"
SOTA_LOU = "/content/drive/MyDrive/sota_lou"

os.environ.update({
    "HF_ENDPOINT": "https://huggingface.co",
    "MODEL_ROOT": f"{SOTA_LOU}/models",
    "DATA_ROOT": f"{SOTA_LOU}/data",
    "DUMP_ROOT": f"{SOTA_LOU}/dump",
    "STEER_OUT_ROOT": f"{SOTA_LOU}/steer_result",
    "MODEL_KEY": "llama3-8b-it",   # dump·steer가 같은 키를 써야 함 (README 참고)
    "LAYER": "10",
    "GPU": "0",
    "ALPHA_MAX": "3",
    "ALPHA_STEP": "0.2",
    "SPLIT_SEED": "42",
})
for k in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE"):
    os.environ.pop(k, None)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # 값은 출력하지 않음

mem_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
BATCH_SIZE = 64 if mem_gb >= 70 else 32 if mem_gb >= 35 else 8
os.environ["BATCH_SIZE"] = str(BATCH_SIZE)
for k in ("MODEL_ROOT", "DATA_ROOT", "DUMP_ROOT", "STEER_OUT_ROOT"):
    os.makedirs(os.environ[k], exist_ok=True)
os.makedirs(f"{SOTA_LOU}/logs", exist_ok=True)
print(f"GPU {torch.cuda.get_device_name(0)} {mem_gb:.0f} GB -> BATCH_SIZE={BATCH_SIZE}")
print({k: os.environ[k] for k in ("MODEL_ROOT", "DATA_ROOT", "DUMP_ROOT", "STEER_OUT_ROOT")})

GPU NVIDIA A100-SXM4-80GB 79 GB -> BATCH_SIZE=64
{'MODEL_ROOT': '/content/drive/MyDrive/sota_lou/models', 'DATA_ROOT': '/content/drive/MyDrive/sota_lou/data', 'DUMP_ROOT': '/content/drive/MyDrive/sota_lou/dump', 'STEER_OUT_ROOT': '/content/drive/MyDrive/sota_lou/steer_result'}


## 4. 저장소 clone (`sota/lou`, submodule 포함)

In [4]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--recurse-submodules", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "submodule", "update", "--init", "--recursive"], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
print("upstream:", subprocess.run(["git", "-C", f"{LOU_DIR}/upstream", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)

e2f9d51 Add Colab notebook and scripts for Lou et al. NQ-Swap reproduction

upstream: 48148daf1179f9cfe201fd812c7dbbdc623aa74f



## 5. 패키지 설치

upstream README는 `torch transformers tokenizers numpy matplotlib`만 요구하고 버전을 고정하지 않습니다. 여기서는 Llama 3.1을 지원하고 upstream 코드(`torch_dtype=`, tuple 출력 hook)와 맞는 4.x 버전을 고정합니다. `device_map`에는 `accelerate`가 필요합니다.

In [5]:
TRANSFORMERS_SPEC = "transformers==4.46.3"
!pip install -q "{TRANSFORMERS_SPEC}" accelerate datasets huggingface_hub numpy matplotlib
import importlib.metadata as md
print({p: md.version(p) for p in ("torch", "transformers", "tokenizers", "accelerate", "datasets", "huggingface_hub")})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
{'torch': '2.14.1+cu130', 'transformers': '4.46.3', 'tokenizers': '0.20.3', 'accelerate': '1.15.0', 'datasets': '4.8.5', 'huggingface_hub': '0.36.2'}


### 실행 helper (출력 스트리밍 + Drive 로그)

In [6]:
import subprocess, sys, time
def run(cmd, **env):
    log = f"{SOTA_LOU}/logs/{time.strftime('%Y%m%d-%H%M%S')}_{os.path.basename(cmd[1]).split('.')[0]}.log"
    e = {**os.environ, **{k: str(v) for k, v in env.items()}}
    with open(log, "w") as f, subprocess.Popen(cmd, env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
        for line in p.stdout:
            sys.stdout.write(line); f.write(line)
    print(f"[exit {p.returncode}] log: {log}")
    if p.returncode:
        raise RuntimeError(f"failed: {' '.join(cmd)}")

## 6. 모델 다운로드 → `$MODEL_ROOT/Meta-Llama-3.1-8B-Instruct`

In [7]:
run(["bash", f"{LOU_DIR}/scripts/download_model.sh"])

[download_model] meta-llama/Llama-3.1-8B-Instruct@main -> /content/drive/MyDrive/sota_lou/models/Meta-Llama-3.1-8B-Instruct (endpoint https://huggingface.co)

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:805: UserWarning: Not enough free disk space to download the file. The expected file size is: 4976.70 MB. The target location /content/drive/MyDrive/sota_lou/models/Meta-Llama-3.1-8B-Instruct/.cache/huggingface/download only has 1407.13 MB free disk space.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:805: UserWarning: Not enough free disk space to download the file. The expected file size is: 4976.70 MB. The target location /content/drive/MyDrive/sota_lou/models/Meta-Llama-3.1-8B-Instruct only has 1407.13 MB free disk space.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:805: UserWarning: Not enough free disk space 

## 7. NQ-Swap 준비 → `$DATA_ROOT/NQ-Swap/dev.jsonl`

In [8]:
run(["bash", f"{LOU_DIR}/scripts/prepare_nqswap.sh"])


Generating dev split: 100%|██████████| 4746/4746 [00:00<00:00, 54853.04 examples/s]
[prepare_nqswap] wrote 4746 rows -> /content/drive/MyDrive/sota_lou/data/NQ-Swap/dev.jsonl
{"repo": "pminervini/NQ-Swap", "revision": "3801dbf037c6f58b0e79246f2a8073ebe5656dad", "split": "dev", "num_rows": 4746, "sha256": "72571fea1a20e8f1ec7d8e2b30d974815a8fac39ee616c8501bc659be2802edb"}
[exit 0] log: /content/drive/MyDrive/sota_lou/logs/20261126-070231_prepare_nqswap.log


## 8. 실행

`run_nqswap_llama.sh`를 단계별로 호출합니다. 각 단계는 완료 산출물이 있으면 건너뜁니다. 다시 계산하려면 `FORCE=1`을 넘기세요.

### 8-1. dump (stage1 closed-book · stage2 with context · 전 layer last-token resid)

In [9]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [13]:
run(["bash", f"{LOU_DIR}/scripts/run_nqswap_llama.sh"], STEPS="dump", BATCH_SIZE="16")

[run_nqswap_llama 07:46:30] model=llama3-8b-it layer=10 batch=16 alpha=±3/0.2 steps=dump
[run_nqswap_llama 07:46:30] 1/4 dump -> /content/drive/MyDrive/sota_lou/dump/llama3-8b-it/NQ-Swap/dev
[run_dump_all_datasets] ROOT=/content/Graduation-Project/SOTA/lou/upstream OUT_ROOT=/content/drive/MyDrive/sota_lou/dump
[run_dump_all_datasets] one combined dump per category (splits at steer time)
[run_dump_all_datasets] NQ-Swap: /content/drive/MyDrive/sota_lou/data/NQ-Swap/dev.jsonl (category=dev)
[run] llama3-8b-it | NQ-Swap | dev
I0000 00:00:1795679195.436867   13736 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795679195.510911   13736 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the followi

### 8-2. 방향 추정(train 70%) + α⁺ 탐색(val 15%, 0..3) + test 평가 → Acc_par

In [16]:
from huggingface_hub import hf_hub_download
import os, re
dst = "/content/models/Meta-Llama-3.1-8B-Instruct"
rev = re.findall(r"[0-9a-f]{40}", open(f"{dst}/.hf_revision").read())
rev = rev[0] if rev else None
print("revision:", rev)
hf_hub_download("meta-llama/Llama-3.1-8B-Instruct", "model-00003-of-00004.safetensors",
                local_dir=dst, revision=rev, token=os.environ["HF_TOKEN"])
!ls -la {dst}/*.safetensors

revision: 0e9e39f249a16976918f6564b8830bc894c89659


model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

-rw------- 1 root root 4976698672 Nov 26 07:02 /content/models/Meta-Llama-3.1-8B-Instruct/model-00001-of-00004.safetensors
-rw------- 1 root root 4999802720 Nov 26 07:02 /content/models/Meta-Llama-3.1-8B-Instruct/model-00002-of-00004.safetensors
-rw-r--r-- 1 root root 4915916176 Nov 26 10:01 /content/models/Meta-Llama-3.1-8B-Instruct/model-00003-of-00004.safetensors
-rw------- 1 root root 1168138808 Nov 26 07:01 /content/models/Meta-Llama-3.1-8B-Instruct/model-00004-of-00004.safetensors


In [17]:
import os, shutil
src = "/content/drive/MyDrive/sota_lou/models/Meta-Llama-3.1-8B-Instruct"
dst = "/content/models/Meta-Llama-3.1-8B-Instruct"
if not os.path.exists(f"{dst}/model-00004-of-00004.safetensors"):
    shutil.copytree(src, dst, dirs_exist_ok=True)
!ls -la {dst}
os.environ["MODEL_ROOT"] = "/content/models"
os.environ["BATCH_SIZE"] = "16"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

run(["bash", f"{LOU_DIR}/scripts/run_nqswap_llama.sh"], STEPS="par")

total 15693148
drwx------ 4 root root       4096 Nov 26 10:01 .
drwxr-xr-x 3 root root       4096 Nov 26 10:01 ..
drwx------ 3 root root       4096 Nov 26 07:01 .cache
-rw------- 1 root root        855 Nov 26 07:01 config.json
-rw------- 1 root root        184 Nov 26 07:01 generation_config.json
-rw------- 1 root root         74 Nov 26 07:02 .hf_revision
-rw------- 1 root root 4976698672 Nov 26 07:02 model-00001-of-00004.safetensors
-rw------- 1 root root 4999802720 Nov 26 07:02 model-00002-of-00004.safetensors
-rw-r--r-- 1 root root 4915916176 Nov 26 10:01 model-00003-of-00004.safetensors
-rw------- 1 root root 1168138808 Nov 26 07:01 model-00004-of-00004.safetensors
-rw------- 1 root root      23950 Nov 26 07:01 model.safetensors.index.json
drwx------ 2 root root       4096 Nov 26 07:01 original
-rw------- 1 root root        296 Nov 26 07:01 special_tokens_map.json
-rw------- 1 root root      55351 Nov 26 07:01 tokenizer_config.json
-rw------- 1 root root    9085657 Nov 26 07:01 toke

### 8-3. α⁻ 탐색(val, −3..−0.2, Acc_ctx 기준) + test 평가 → Acc_ctx

In [18]:
run(["bash", f"{LOU_DIR}/scripts/run_nqswap_llama.sh"], STEPS="ctx")

[run_nqswap_llama 11:58:38] model=llama3-8b-it layer=10 batch=16 alpha=±3/0.2 steps=ctx
[run_nqswap_llama 11:58:38] 3/4 alpha- search (-3..-0.2) on val -> /content/drive/MyDrive/sota_lou/steer_result/ctx_sweep
[train_val_test] same_dataset split n=2331 train=1632 val=350 test=349 (seed=42, train_ratio=0.7, val_ratio=0.15)
I0000 00:00:1795694334.525371   78428 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795694334.592829   78428 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795694337.096893   78428 port.cc:153] oneDNN custom operations are 

### 8-4. Unsteered test (α=0)

In [19]:
run(["bash", f"{LOU_DIR}/scripts/run_nqswap_llama.sh"], STEPS="base")

[run_nqswap_llama 13:57:07] model=llama3-8b-it layer=10 batch=16 alpha=±3/0.2 steps=base
[run_nqswap_llama 13:57:07] 4/4 unsteered test (alpha=0) -> /content/drive/MyDrive/sota_lou/steer_result/base
[train_val_test] same_dataset split n=2331 train=1632 val=350 test=349 (seed=42, train_ratio=0.7, val_ratio=0.15)
I0000 00:00:1795701442.777422  108857 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795701442.845589  108857 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795701445.371547  108857 port.cc:153] oneDNN custom operations are on. You may

## 9. Table 1 비교

In [20]:
run(["bash", f"{LOU_DIR}/scripts/run_nqswap_llama.sh"], STEPS="summary")
import json
s = json.load(open(f"{os.environ['STEER_OUT_ROOT']}/table1_summary.json"))
print(json.dumps({k: s[k] for k in ("alpha_plus", "alpha_plus_if_selected_by_val_acc_par", "alpha_minus")}, indent=2))

[run_nqswap_llama 14:13:52] model=llama3-8b-it layer=10 batch=16 alpha=±3/0.2 steps=summary
NQ-Swap test, llama3-8b-it, layer 10, n_test=349
                              Acc_par  Acc_ctx           paper
Unsteered (alpha=0)              32.4     48.7    33.7    49.4
Steer par (alpha=2.4)            36.1             56.8        
Steer ctx (alpha=-0.8)                    54.2            55.5
[note] val Acc_par/N would select alpha+=1.6 instead of 2.4
-> /content/drive/MyDrive/sota_lou/steer_result/table1_summary.json

NQ-Swap test re-score (n_test=349); upstream = first sentence, 'correct' includes both | paper = full output (<=200 tok), App. C
run                       up par  up ctx  pp par  pp ctx  par-only  ctx-only   both  neither        Table1
unsteered (a=0)             32.4    48.7    17.2    28.7      17.2      28.7   47.6      6.6     33.7/49.4
steer_par (a=2.4)           36.1     3.4    36.1     2.9      36.1       2.9    2.6     58.5      par 56.8
steer_ctx (a=-0.8)          

In [21]:
# ## 10. 추가 실험: α⁺=1.6 (val Acc_par/N 기준이었다면 선택됐을 값)
import os, glob, json, sys

A = "1.6"
D = f"{os.environ['DUMP_ROOT']}/{os.environ['MODEL_KEY']}/NQ-Swap/dev"
OUT = f"{os.environ['STEER_OUT_ROOT']}/par_a{A}"
SRC = f"{LOU_DIR}/upstream/src/run_resid_projection_ablation.py"

if not glob.glob(f"{OUT}/**/L10_alpha{A}_steer_result.json", recursive=True):
    run([sys.executable, SRC,
         "--eval-mode", "train_val_test", "--split-protocol", "same_dataset", "--split-seed", "42",
         "--train-ratio", "0.7", "--val-ratio", "0.15", "--model", os.environ["MODEL_KEY"],
         "--npz", f"{D}/resid_hidden_state__stage1_correct_use_wrong_context.npz",
         "--test-json", f"{D}/internal_aware_data__stage1_correct_use_wrong_context.json",
         "--out-root", OUT, "--output-category", "dev",
         "--direction-category-tag", "dev", "--test-category-tag", "dev",
         "--gpu", "0", "--layers", "10",
         "--alpha-start", A, "--alpha-end", A, "--alpha-step", "0.2",
         "--batch-size", os.environ["BATCH_SIZE"],
         "--max-samples-stage1", "0", "--max-samples-stage2", "0",
         "--stage1-min-pass-rate", "-1"])

# 논문 방식 재채점 (base·ctx는 기존 결과, par만 α=1.6으로 교체)
R = os.environ["STEER_OUT_ROOT"]
f = lambda p: sorted(glob.glob(p, recursive=True))[-1]
par = f(f"{OUT}/**/L10_alpha{A}_steer_result.json")
base = f(f"{R}/base/**/L10_alpha0*_steer_result.json")
ctx = f(f"{R}/ctx/**/L10_alpha-0.8_steer_result.json")
print("par :", par, "\nbase:", base, "\nctx :", ctx)
run([sys.executable, f"{LOU_DIR}/scripts/rescore_paper_metric.py", "--base", base, "--par", par, "--ctx", ctx])

[train_val_test] same_dataset split n=2331 train=1632 val=350 test=349 (seed=42, train_ratio=0.7, val_ratio=0.15)
I0000 00:00:1795702608.170858  113926 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795702608.238119  113926 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795702610.764820  113926 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:

RuntimeError: failed: /usr/bin/python3 /content/Graduation-Project/SOTA/lou/scripts/rescore_paper_metric.py --base /content/drive/MyDrive/sota_lou/steer_result/base/llama3-8b-it/NQ-Swap/dev__dir_dev__test_dev/L10_alpha0_steer_result.json --par /content/drive/MyDrive/sota_lou/steer_result/par_a1.6/llama3-8b-it/NQ-Swap/dev__dir_dev__test_dev/L10_alpha1.6_steer_result.json --ctx /content/drive/MyDrive/sota_lou/steer_result/ctx/llama3-8b-it/NQ-Swap/dev__dir_dev__test_dev/L10_alpha-0.8_steer_result.json

In [22]:
import shutil, os
# 기존 요약(table1_summary.json)을 덮어쓰지 않도록 par_a1.6 폴더를 steer-root로 쓴다
main_sum = f"{R}/table1_summary.json"
alt_sum = f"{OUT}/table1_summary.json"
if os.path.exists(main_sum) and not os.path.exists(alt_sum):
    shutil.copy(main_sum, alt_sum)

run([sys.executable, f"{LOU_DIR}/scripts/rescore_paper_metric.py",
     "--steer-root", OUT,
     "--manifest", f"{D}/internal_aware_data__stage1_correct_use_wrong_context.json",
     "--base", base, "--par", par, "--ctx", ctx])


NQ-Swap test re-score (n_test=349); upstream = first sentence, 'correct' includes both | paper = full output (<=200 tok), App. C
run                       up par  up ctx  pp par  pp ctx  par-only  ctx-only   both  neither        Table1
unsteered (a=0)             32.4    48.7    17.2    28.7      17.2      28.7   47.6      6.6     33.7/49.4
steer_par (a=1.6)           47.6     7.7    49.6     6.9      49.6       6.9    4.6     39.0      par 56.8
steer_ctx (a=-0.8)          14.6    54.2     9.7    47.3       9.7      47.3   23.2     19.8      ctx 55.5

[aux, not the reported metric] word-boundary match (\b) on the same normalized text; rows with a normalized answer of <= 3 chars
run                       wb par  wb ctx  par-only  ctx-only   both  neither  short any   par   ctx
unsteered (a=0)             17.2    28.4      17.2      28.4   47.6      6.9         18    14    11
steer_par (a=1.6)           49.6     6.9      49.6       6.9    4.6     39.0         18    14    11
steer_ctx (a